# LongMemEval A0 Full Benchmark Kaggle Pipeline

This notebook runs **Step 1 / A0: BM25 session baseline** on `longmemeval_s_cleaned.json` with a Kaggle-friendly CPU retrieval + OpenAI-compatible API pipeline:

1. Prepare the LongMemEval source tree.
2. Download or import the cleaned benchmark data.
3. Run BM25 retrieval at session granularity.
4. Run retrieval-augmented generation through an OpenAI-compatible chat-completions API.
5. Run LLM-as-judge QA evaluation with visible progress.
6. Export normalized A0 artifacts under `results/`.

Default run: all examples, CPU BM25 retrieval, top-5 retrieved sessions, and API-based generation/evaluation through 9router/OpenAI-compatible chat completions.

## Kaggle requirements

- Internet must be enabled unless you attach the benchmark JSON files as a Kaggle Dataset.
- Add `OPENAI_API_KEY` in Kaggle Add-ons -> Secrets.
- Add `OPENAI_BASE_URL` as a Kaggle Secret when using a router/OpenAI-compatible endpoint.
- The notebook never prints the API key and does not pass it as a command-line argument.
- Full benchmark cost is roughly one generation call plus one LLM-as-judge call per example. Run a small smoke test first by setting `RUN_FULL=0` and `N_EXAMPLES=3`.

## A0 outputs

The main result cell writes:

- `results/A0_bm25_session_lme_s_cleaned_hypotheses.jsonl`
- `results/A0_bm25_session_lme_s_cleaned_eval_log.jsonl`
- `results/A0_bm25_session_lme_s_cleaned_retrieval_log.jsonl`
- `results/A0_bm25_session_lme_s_cleaned_summary.json`
- `results/A0_bm25_session_lme_s_cleaned_failed_cases.csv`
- `results/A0_bm25_session_lme_s_cleaned_cost_latency.csv`


In [1]:
from pathlib import Path
import csv
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import urllib.request

ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
PROJECT_DIR_NAME = os.getenv('PROJECT_DIR_NAME', 'LongMemEval-Experiment')
REPO_URL = os.getenv('LONGMEMEVAL_REPO', 'https://github.com/toanthangO20/LongMemEval-Experiment.git')
CHECKOUT_DIR = os.getenv('LONGMEMEVAL_CHECKOUT_DIR', PROJECT_DIR_NAME)

# Choose the benchmark file. A0 defaults to LongMemEval-S cleaned.
DATASET_NAME = os.getenv('LONGMEMEVAL_DATASET', 'longmemeval_s_cleaned.json')

# Full-benchmark defaults. Set RUN_FULL=0 and lower N_EXAMPLES only for debugging.
RUN_FULL = os.getenv('RUN_FULL', '1') == '1'
N_EXAMPLES = int(os.getenv('N_EXAMPLES', '500'))
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '7'))

# Retrieval configuration for A0.
RETRIEVER = os.getenv('RETRIEVER', 'flat-bm25')
GRANULARITY = os.getenv('GRANULARITY', 'session')
TOPK_CONTEXT = int(os.getenv('TOPK_CONTEXT', '5'))
REPORT_TOPKS = [1, 3, 5, 10, 20]
BM25_NUM_PROCESSES = int(os.getenv('BM25_NUM_PROCESSES', '4'))

# Keep full-run notebook output readable. Artifacts still contain all records.
MAX_NOTEBOOK_PREVIEW = int(os.getenv('MAX_NOTEBOOK_PREVIEW', '25'))

# 9router/OpenAI-compatible reader and evaluator configuration.
DEFAULT_OPENAI_BASE_URL = os.getenv('DEFAULT_OPENAI_BASE_URL', 'https://splashed-nastily-stopped.ngrok-free.dev/v1')
GEN_MODEL_NAME = os.getenv('GEN_MODEL_NAME', 'cx/gpt-5.2')
GEN_MODEL_ALIAS = os.getenv('GEN_MODEL_ALIAS', 'router-gpt-5.2')
METRIC_MODEL_SHORT = os.getenv('METRIC_MODEL_SHORT', 'router-gpt-5.2')
METRIC_MODEL_NAME = os.getenv('METRIC_MODEL_NAME', 'cx/gpt-5.2')
MODEL_MAX_LENGTH = int(os.getenv('MODEL_MAX_LENGTH', '128000'))
GEN_LENGTH = int(os.getenv('GEN_LENGTH', '300'))
HISTORY_FORMAT = os.getenv('HISTORY_FORMAT', 'json')
USERONLY = os.getenv('USERONLY', 'false')
OPENAI_DEFAULT_HEADERS = os.getenv('OPENAI_DEFAULT_HEADERS', '{"ngrok-skip-browser-warning":"true"}')

_DATASET_SLUGS = {
    'longmemeval_s_cleaned.json': 'lme_s_cleaned',
    'longmemeval_m_cleaned.json': 'lme_m_cleaned',
    'longmemeval_oracle.json': 'lme_oracle',
}
DATASET_SLUG = _DATASET_SLUGS.get(DATASET_NAME, Path(DATASET_NAME).stem.replace('-', '_'))
RETRIEVER_SLUG = RETRIEVER.replace('flat-', '').replace('-', '_')
EXPERIMENT_ID = os.getenv('EXPERIMENT_ID', f'A0_{RETRIEVER_SLUG}_{GRANULARITY}_{DATASET_SLUG}')

print('Working root:', ROOT)
print('Dataset:', DATASET_NAME)
print('Experiment ID:', EXPERIMENT_ID)
print('Run full benchmark:', RUN_FULL, '| N_EXAMPLES:', N_EXAMPLES)
print('Retriever:', RETRIEVER, '| Granularity:', GRANULARITY)
print('Generation model:', GEN_MODEL_NAME)
print('Metric model alias:', METRIC_MODEL_SHORT, '| metric model:', METRIC_MODEL_NAME)
print('Top-k context:', TOPK_CONTEXT, '| Report top-k:', REPORT_TOPKS)
print('Generation max tokens:', GEN_LENGTH)
print('BM25 processes:', BM25_NUM_PROCESSES)


Working root: /kaggle/working
Dataset: longmemeval_s_cleaned.json
Experiment ID: A0_bm25_session_lme_s_cleaned
Run full benchmark: True | N_EXAMPLES: 500
Retriever: flat-bm25 | Granularity: session
Generation model: cx/gpt-5.2
Metric model alias: router-gpt-5.2 | metric model: cx/gpt-5.2
Top-k context: 5 | Report top-k: [1, 3, 5, 10, 20]
Generation max tokens: 300
BM25 processes: 4


## Install lightweight dependencies

The full project requirements include vLLM and heavier CUDA packages. For this BM25 + API pipeline, these packages are enough. `httpx==0.27.2` is pinned for compatibility with `openai==1.35.1`. `tqdm` is installed explicitly because long Kaggle cells use progress bars.


In [2]:
%pip install -q openai==1.35.1 httpx==0.27.2 backoff==2.2.1 rank-bm25==0.2.2 tiktoken==0.7.0 sentence-transformers==2.7.0 scikit-learn tqdm==4.66.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.8/326.8 kB 7.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 87.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 11.9 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not inst

In [3]:
import httpx
import openai
print('openai version:', openai.__version__)
print('httpx version:', httpx.__version__)
assert tuple(map(int, httpx.__version__.split('.')[:2])) < (0, 28), 'httpx must be < 0.28 for openai==1.35.1'


openai version: 1.35.1
httpx version: 0.27.2


## Load secrets and helper functions

Secrets are read from environment variables first, then from Kaggle Secrets. The API key is only stored in the process environment and is not passed to subprocess command lines.


In [4]:
def run_cmd(cmd, cwd=None, env=None, check=True, log_file=None):
    shown = [str(x) for x in cmd]
    print('$', ' '.join(shown))
    run_kwargs = {
        'cwd': str(cwd) if cwd else None,
        'env': env,
        'check': check,
        'text': True,
    }
    if log_file:
        log_file = Path(log_file)
        log_file.parent.mkdir(parents=True, exist_ok=True)
        print('Writing command output to:', log_file)
        with log_file.open('w', encoding='utf-8') as stream:
            try:
                return subprocess.run(cmd, stdout=stream, stderr=subprocess.STDOUT, **run_kwargs)
            except subprocess.CalledProcessError:
                print('Command failed. Last log lines:')
                lines = log_file.read_text(encoding='utf-8', errors='replace').splitlines()
                for line in lines[-80:]:
                    print(line)
                raise
    return subprocess.run(cmd, **run_kwargs)


def load_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ''


OPENAI_API_KEY = load_secret('OPENAI_API_KEY')
OPENAI_ORGANIZATION = load_secret('OPENAI_ORGANIZATION')
OPENAI_BASE_URL = load_secret('OPENAI_BASE_URL') or os.getenv('OPENAI_BASE_URL', DEFAULT_OPENAI_BASE_URL)

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if OPENAI_ORGANIZATION:
    os.environ['OPENAI_ORGANIZATION'] = OPENAI_ORGANIZATION
if OPENAI_BASE_URL:
    os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_DEFAULT_HEADERS'] = OPENAI_DEFAULT_HEADERS
os.environ['TOKENIZER_BACKEND'] = os.getenv('TOKENIZER_BACKEND', 'openai')
os.environ['MODEL_MAX_LENGTH'] = str(MODEL_MAX_LENGTH)
os.environ['METRIC_MODEL_NAME'] = METRIC_MODEL_NAME

print('OPENAI_API_KEY configured:', bool(OPENAI_API_KEY))
print('OPENAI_ORGANIZATION configured:', bool(OPENAI_ORGANIZATION))
print('OPENAI_BASE_URL configured:', bool(OPENAI_BASE_URL))
print('MODEL_MAX_LENGTH:', MODEL_MAX_LENGTH)
print('TOKENIZER_BACKEND:', os.environ['TOKENIZER_BACKEND'])

OPENAI_API_KEY configured: True
OPENAI_ORGANIZATION configured: False
OPENAI_BASE_URL configured: True
MODEL_MAX_LENGTH: 128000
TOKENIZER_BACKEND: openai


## Prepare the source tree

When the notebook is run from a repository checkout, it uses that checkout. Otherwise it clones this public experiment repository into `/kaggle/working`. Set `LONGMEMEVAL_REPO` only if you intentionally want to test another fork.


In [6]:
def looks_like_longmemeval_repo(path):
    path = Path(path)
    return (path / 'src' / 'retrieval' / 'run_retrieval.py').exists() and (path / 'src' / 'generation' / 'run_generation.py').exists()


candidate_dirs = [Path.cwd(), ROOT / CHECKOUT_DIR, ROOT / 'LongMemEval', ROOT / PROJECT_DIR_NAME]
REPO_DIR = None
for candidate in candidate_dirs:
    if looks_like_longmemeval_repo(candidate):
        REPO_DIR = candidate.resolve()
        break

if REPO_DIR is None:
    REPO_DIR = (ROOT / CHECKOUT_DIR).resolve()
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    if not looks_like_longmemeval_repo(REPO_DIR):
        raise RuntimeError(f'Checkout does not look like a LongMemEval repo: {REPO_DIR}')

RESULTS_DIR = REPO_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Using source tree:', REPO_DIR)
print('A0 results directory:', RESULTS_DIR)
print('Top-level files:')
for path in sorted(REPO_DIR.iterdir()):
    if path.name != '.git':
        print(' -', path.name)


$ git clone --depth 1 https://github.com/toanthangO20/LongMemEval-Experiment.git /kaggle/working/LongMemEval-Experiment


Cloning into '/kaggle/working/LongMemEval-Experiment'...


Using source tree: /kaggle/working/LongMemEval-Experiment
A0 results directory: /kaggle/working/LongMemEval-Experiment/results
Top-level files:
 - .gitignore
 - LICENSE
 - README.md
 - assets
 - data
 - notebooks
 - requirements-full.txt
 - requirements-lite.txt
 - results
 - src


## Apply Kaggle compatibility patches

These patches are idempotent. They keep the benchmark logic intact while making the scripts safer for Kaggle: no API key in printed args, optional OpenAI-compatible base URL, custom metric model alias support, and NumPy 2.x compatibility for retrieval metrics.


In [9]:
def replace_text(path, old, new):
    text = path.read_text(encoding='utf-8')
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        return True
    return False


gen_py = REPO_DIR / 'src' / 'generation' / 'run_generation.py'
gen_text = gen_py.read_text(encoding='utf-8')
if 'import os\n' not in gen_text[:150]:
    gen_text = gen_text.replace('import sys\n', 'import sys\nimport os\n')
gen_py.write_text(gen_text, encoding='utf-8')

replace_text(
    gen_py,
    "    if args.openai_organization:\n        openai.organization = args.openai_organization\n",
    "    openai_organization = args.openai_organization or os.getenv('OPENAI_ORGANIZATION')\n    if openai_organization:\n        openai.organization = openai_organization\n",
)
replace_text(
    gen_py,
    "    parser.add_argument('--openai_key', type=str, required=True)\n",
    "    parser.add_argument('--openai_key', type=str, default=None)\n",
)
replace_text(
    gen_py,
    "def check_args(args):\n    print(args)\n",
    "def check_args(args):\n    safe_args = argparse.Namespace(**vars(args))\n    if safe_args.openai_key:\n        safe_args.openai_key = '***'\n    if safe_args.openai_organization:\n        safe_args.openai_organization = '***'\n    print(safe_args)\n",
)
replace_text(
    gen_py,
    "    client = OpenAI(\n        api_key=args.openai_key,\n        base_url=args.openai_base_url,\n    )",
    "    openai_key = args.openai_key or os.getenv('OPENAI_API_KEY')\n    if not openai_key:\n        raise RuntimeError('OPENAI_API_KEY is required. Set it in the environment or pass --openai_key.')\n    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    client = OpenAI(\n        api_key=openai_key,\n        base_url=args.openai_base_url,\n        default_headers=default_headers,\n    )",
)
replace_text(
    gen_py,
    "    model_max_length = model2maxlength[args.model_name]\n",
    "    model_max_length = model2maxlength.get(args.model_name, int(os.getenv('MODEL_MAX_LENGTH', '128000')))\n",
)
replace_text(
    gen_py,
    "    if 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
    "    if os.getenv('TOKENIZER_BACKEND', '').lower() == 'openai' or 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
)
replace_text(
    gen_py,
    "            total_prompt_tokens += completion.usage.prompt_tokens\n            total_completion_tokens += completion.usage.completion_tokens\n",
    "            usage = getattr(completion, 'usage', None)\n            total_prompt_tokens += (getattr(usage, 'prompt_tokens', 0) or 0)\n            total_completion_tokens += (getattr(usage, 'completion_tokens', 0) or 0)\n",
)
replace_text(
    gen_py,
    "@backoff.on_exception(backoff.constant, (openai.RateLimitError), \n                      interval=5)\n",
    "@backoff.on_exception(\n    backoff.expo,\n    (openai.RateLimitError, openai.APIError, openai.APIConnectionError, openai.APITimeoutError),\n    max_tries=int(os.getenv('OPENAI_MAX_RETRIES', '8')),\n)\n",
)

retrieval_py = REPO_DIR / 'src' / 'retrieval' / 'run_retrieval.py'
replace_text(retrieval_py, 'for k in [1, 3, 5, 10, 30, 50]:', 'for k in [1, 3, 5, 10, 20, 30, 50]:')
replace_text(
    retrieval_py,
    "    num_processes = torch.cuda.device_count()\n    if 'bm25' in args.retriever:\n        num_processes = 10\n    print('Setting num processes = {} with retriever {}'.format(num_processes, args.retriever))\n    mp.set_start_method('spawn')\n    pool = mp.Pool(num_processes)\n",
    "    num_processes = torch.cuda.device_count()\n    if 'bm25' in args.retriever:\n        num_processes = int(os.getenv('BM25_NUM_PROCESSES', '4'))\n    num_processes = max(1, min(num_processes, len(in_data)))\n    print('Setting num processes = {} with retriever {}'.format(num_processes, args.retriever))\n    try:\n        mp.set_start_method('spawn')\n    except RuntimeError:\n        pass\n    pool = mp.Pool(num_processes)\n",
)
replace_text(
    retrieval_py,
    "    for d in pool.imap_unordered(worker, in_data_chunked):\n        results += d\n",
    "    for d in tqdm(pool.imap_unordered(worker, in_data_chunked), total=len(in_data_chunked), desc='Retrieval chunks'):\n        results += d\n",
)

eval_py = REPO_DIR / 'src' / 'evaluation' / 'evaluate_qa.py'
eval_text = eval_py.read_text(encoding='utf-8')
if 'import os\n' not in eval_text[:150]:
    eval_text = eval_text.replace('import sys\n', 'import sys\nimport os\n')
eval_py.write_text(eval_text, encoding='utf-8')
if "METRIC_MODEL_NAME" not in eval_py.read_text(encoding='utf-8'):
    replace_text(
        eval_py,
        "    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
        "    if metric_model_short not in model_zoo and os.getenv('METRIC_MODEL_NAME'):\n        model_zoo[metric_model_short] = (os.getenv('METRIC_MODEL_NAME'), 'openai')\n    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
    )
replace_text(
    eval_py,
    "        openai_api_base = None\n",
    "        openai_api_base = os.getenv('OPENAI_BASE_URL') or None\n        if not openai_api_key:\n            raise RuntimeError('OPENAI_API_KEY is required for OpenAI-compatible evaluation models.')\n",
)
replace_text(
    eval_py,
    "    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n    )",
    "    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n        default_headers=default_headers,\n    )",
)
replace_text(
    eval_py,
    "@backoff.on_exception(backoff.expo, (openai.RateLimitError,\n                                    openai.APIError))\n",
    "@backoff.on_exception(\n    backoff.expo,\n    (openai.RateLimitError, openai.APIError, openai.APIConnectionError, openai.APITimeoutError),\n    max_tries=int(os.getenv('OPENAI_MAX_RETRIES', '8')),\n)\n",
)
replace_text(eval_py, "            print(json.dumps(entry), file=out_f)\n", "            print(json.dumps(entry), file=out_f, flush=True)\n")

eval_utils_py = REPO_DIR / 'src' / 'retrieval' / 'eval_utils.py'
replace_text(eval_utils_py, 'np.asfarray(relevances)[:k]', 'np.asarray(relevances, dtype=float)[:k]')

print('Compatibility and progress patches applied or already present.')


Compatibility and progress patches applied or already present.


## Fetch benchmark data

The notebook first searches `/kaggle/input` for `DATASET_NAME`. If the file is not attached as a Kaggle Dataset, it downloads the official cleaned benchmark file from Hugging Face.


In [10]:
DATA_DIR = REPO_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
target_file = DATA_DIR / DATASET_NAME

if not target_file.exists():
    candidates = list(Path('/kaggle/input').rglob(DATASET_NAME)) if Path('/kaggle/input').exists() else []
    if candidates:
        print('Copying dataset from Kaggle input:', candidates[0])
        shutil.copy2(candidates[0], target_file)
    else:
        url = f'https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/{DATASET_NAME}'
        print('Downloading:', url)
        urllib.request.urlretrieve(url, target_file)
else:
    print('Dataset already exists:', target_file)

data = json.loads(target_file.read_text(encoding='utf-8'))
print('Loaded examples:', len(data))
print('First example keys:', sorted(data[0].keys()))
counts = {}
for row in data:
    counts[row['question_type']] = counts.get(row['question_type'], 0) + 1
print('Question type counts:')
print(json.dumps(counts, indent=2))


Dataset already exists: /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned.json
Loaded examples: 500
First example keys: ['answer', 'answer_session_ids', 'haystack_dates', 'haystack_session_ids', 'haystack_sessions', 'question', 'question_date', 'question_id', 'question_type']
Question type counts:
{
  "single-session-user": 70,
  "multi-session": 133,
  "single-session-preference": 30,
  "temporal-reasoning": 133,
  "knowledge-update": 78,
  "single-session-assistant": 56
}


## Select the full benchmark file

With `RUN_FULL=True`, this notebook uses all examples in `DATASET_NAME`. Set `RUN_FULL=False` and reduce `N_EXAMPLES` only when debugging API, endpoint, or output-format issues.

In [11]:
if RUN_FULL:
    work_file = target_file
    work_data = data
else:
    rng = random.Random(RANDOM_SEED)
    work_data = data.copy()
    rng.shuffle(work_data)
    work_data = work_data[:N_EXAMPLES]
    sample_name = f'{target_file.stem}_sample{len(work_data)}_seed{RANDOM_SEED}.json'
    work_file = DATA_DIR / sample_name
    work_file.write_text(json.dumps(work_data, ensure_ascii=False), encoding='utf-8')

print('Active benchmark file:', work_file)
print('Active examples:', len(work_data))


Active benchmark file: /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned.json
Active examples: 500


## Step 1: memory retrieval

This runs the repository retrieval code and writes a JSONL retrieval log containing `retrieval_results`. The default is `flat-bm25` over sessions, which is CPU-friendly and mirrors the baseline memory retrieval stage.


In [12]:
retrieval_out_dir = REPO_DIR / 'retrieval_logs' / RETRIEVER / GRANULARITY
retrieval_out_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + env.get('PYTHONPATH', '')
env['BM25_NUM_PROCESSES'] = str(BM25_NUM_PROCESSES)

cmd = [
    sys.executable, 'run_retrieval.py',
    '--in_file', str(work_file),
    '--retriever', RETRIEVER,
    '--granularity', GRANULARITY,
    '--index_expansion_method', 'none',
    '--index_expansion_result_join_mode', 'none',
    '--index_expansion_result_cache', 'none',
    '--out_dir', str(retrieval_out_dir),
    '--outfile_prefix', work_file.name,
    '--cache_dir', str(REPO_DIR / 'model_cache'),
]
print('Retrieval progress is shown by the subprocess tqdm bars below.')
run_cmd(cmd, cwd=REPO_DIR / 'src' / 'retrieval', env=env)

retrieval_log = retrieval_out_dir / f'{work_file.name}_retrievallog_{GRANULARITY}_{RETRIEVER}'
print('Retrieval log:', retrieval_log)
print('Exists:', retrieval_log.exists(), '| Size MB:', round(retrieval_log.stat().st_size / 1e6, 2) if retrieval_log.exists() else None)


Retrieval progress is shown by the subprocess tqdm bars below.
$ /usr/bin/python3 run_retrieval.py --in_file /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned.json --retriever flat-bm25 --granularity session --index_expansion_method none --index_expansion_result_join_mode none --index_expansion_result_cache none --out_dir /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session --outfile_prefix longmemeval_s_cleaned.json --cache_dir /kaggle/working/LongMemEval-Experiment/model_cache
Namespace(in_file='/kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned.json', out_dir='/kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session', outfile_prefix='longmemeval_s_cleaned.json', cache_dir='/kaggle/working/LongMemEval-Experiment/model_cache', retriever='flat-bm25', granularity='session', index_expansion_method='none', index_expansion_llm=None, index_expansion_result_cache='none', index_expansion_result_join_mode='none')
Setting num proc

Retrieval chunks: 100%|██████████| 4/4 [00:15<00:00,  3.89s/it]


Ignored 30 instances due to abstention: {'88432d0a_abs', 'ba358f49_abs', '2698e78f_abs', '6456829e_abs', '0862e8bf_abs', '6aeb4375_abs', '2311e44b_abs', '031748ae_abs', 'a96c20ee_abs', 'gpt4_372c3eed_abs', 'gpt4_c27434e8_abs', 'f4f1d8a4_abs', '60bf93ed_abs', '80ec1f4f_abs', 'gpt4_70e84552_abs', '19b5f2b3_abs', 'e5ba910e_abs', 'f685340e_abs', 'eeda8a6d_abs', 'bc8a6e93_abs', '15745da0_abs', '0ddfec37_abs', '29f2956b_abs', '2133c1b5_abs', '982b5123_abs', 'gpt4_fe651585_abs', 'gpt4_93159ced_abs', '09ba9854_abs', 'edced276_abs', 'c8090214_abs'}
Additionally ignored 51 instances due to no target turns from the user side: {'71a3fd6b', '8464fc84', 'ac031881', 'c7cf7dfd', '4baee567', '51b23612', '89527b6b', 'ceb54acb', '18dcd5a5', '41275add', 'eaca4986', 'e48988bc', 'b759caee', '488d3006', '28bcfaac', 'c8f1aeed', '7a8d0b71', '778164c6', '70b3e69b', 'e8a79c70', '6ae235be', '16c90bf4', 'e3fc4d6e', '3249768e', 'cc539528', '3e321797', 'e982271f', 'e9327a54', '8b9d4367', '1568498a', '1d4da289', '8ae

In [13]:
run_cmd([sys.executable, 'src/evaluation/print_retrieval_metrics.py', str(retrieval_log)], cwd=REPO_DIR, env=env)


$ /usr/bin/python3 src/evaluation/print_retrieval_metrics.py /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25
Session-level metrics:
	recall_all@5 = 0.7702, 	ndcg_any@5 = 0.6878, 	recall_all@10 = 0.8426, 	ndcg_any@10 = 0.7078
Turn-level metrics:


CompletedProcess(args=['/usr/bin/python3', 'src/evaluation/print_retrieval_metrics.py', '/kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25'], returncode=0)

## Step 2: retrieval-augmented generation

This stage reads the retrieval log and calls an OpenAI-compatible chat completions API. The API key is read by `run_generation.py` from `OPENAI_API_KEY`, so it is not printed in notebook output.


In [14]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set Kaggle Secret OPENAI_API_KEY before running generation/evaluation cells.')

from tqdm.auto import tqdm

run_id = time.strftime('%Y%m%d-%H%M%S')
generation_out_dir = REPO_DIR / 'generation_logs' / f'{RETRIEVER}-{GRANULARITY}' / GEN_MODEL_ALIAS / 'con'
generation_out_dir.mkdir(parents=True, exist_ok=True)

if RETRIEVER == 'oracle':
    retriever_type = f'oracle-{GRANULARITY}'
else:
    retriever_type = f'flat-{GRANULARITY}'

suffix = f'_{run_id}_kaggle'
cmd = [
    sys.executable, 'run_generation.py',
    '--in_file', str(retrieval_log),
    '--out_dir', str(generation_out_dir),
    '--out_file_suffix', suffix,
    '--model_name', GEN_MODEL_NAME,
    '--model_alias', GEN_MODEL_ALIAS,
    '--retriever_type', retriever_type,
    '--merge_key_expansion_into_value', 'none',
    '--topk_context', str(TOPK_CONTEXT),
    '--history_format', HISTORY_FORMAT,
    '--gen_length', str(GEN_LENGTH),
    '--useronly', USERONLY,
    '--cot', 'true',
    '--con', 'false',
]
if OPENAI_BASE_URL:
    cmd.extend(['--openai_base_url', OPENAI_BASE_URL])


def count_lines(path):
    if not path or not Path(path).exists():
        return 0
    with Path(path).open(encoding='utf-8', errors='replace') as f:
        return sum(1 for _ in f)


def count_occurrences(path, needle):
    if not path or not Path(path).exists():
        return 0
    count = 0
    with Path(path).open(encoding='utf-8', errors='replace') as f:
        for line in f:
            if needle in line:
                count += 1
    return count


expected_examples = count_lines(retrieval_log)
generation_stdout_log = generation_out_dir / f'run_generation{suffix}.stdout.log'
print('$', ' '.join(str(x) for x in cmd))
print('Writing command output to:', generation_stdout_log)
print('Expected examples:', expected_examples)

generation_stdout_log.parent.mkdir(parents=True, exist_ok=True)
generation_start_time = time.time()
with generation_stdout_log.open('w', encoding='utf-8') as stream:
    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO_DIR / 'src' / 'generation'),
        env=env,
        stdout=stream,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pbar = tqdm(total=expected_examples, desc='Generation', unit='example')
    last_done = 0
    last_report_ts = 0
    hyp_file = None

    while True:
        hyp_files = sorted(generation_out_dir.glob(f'*{suffix}'), key=lambda p: p.stat().st_mtime)
        if hyp_files:
            hyp_file = hyp_files[-1]

        successful = count_lines(hyp_file) if hyp_file else 0
        failed = count_occurrences(generation_stdout_log, 'One exception captured')
        done = min(successful + failed, expected_examples)
        if done > last_done:
            pbar.update(done - last_done)
            last_done = done

        now = time.time()
        if now - last_report_ts >= 30:
            percent = (done / expected_examples * 100) if expected_examples else 0
            elapsed = now - generation_start_time
            rate = done / elapsed if elapsed > 0 else 0
            remaining = ((expected_examples - done) / rate) if rate > 0 else None
            eta = f'{remaining/60:.1f} min' if remaining is not None else 'unknown'
            print(
                f'Progress: {done}/{expected_examples} ({percent:.1f}%) | '
                f'success={successful} | failed={failed} | '
                f'elapsed={elapsed/60:.1f} min | ETA={eta}'
            )
            if hyp_file:
                print('Current hypothesis file:', hyp_file)
            print('Stdout log:', generation_stdout_log)
            last_report_ts = now

        if proc.poll() is not None:
            break
        time.sleep(5)

    successful = count_lines(hyp_file) if hyp_file else 0
    failed = count_occurrences(generation_stdout_log, 'One exception captured')
    done = min(successful + failed, expected_examples)
    if done > last_done:
        pbar.update(done - last_done)
    pbar.close()

generation_latency_seconds = time.time() - generation_start_time
return_code = proc.returncode
if return_code != 0:
    print('Generation command failed. Last log lines:')
    lines = generation_stdout_log.read_text(encoding='utf-8', errors='replace').splitlines()
    for line in lines[-80:]:
        print(line)
    raise RuntimeError(f'run_generation.py failed with exit code {return_code}')

hyp_files = sorted(generation_out_dir.glob(f'*{suffix}'), key=lambda p: p.stat().st_mtime)
if not hyp_files:
    raise FileNotFoundError(f'No generation output found with suffix {suffix}')
hyp_file = hyp_files[-1]
successful_lines = count_lines(hyp_file)
failed_examples = count_occurrences(generation_stdout_log, 'One exception captured')
if successful_lines != expected_examples:
    raise RuntimeError(
        f'Generation incomplete: expected {expected_examples} hypotheses, '
        f'got {successful_lines}; failed examples reported: {failed_examples}. '
        f'Inspect {generation_stdout_log} before running evaluation.'
    )

print('Hypothesis file:', hyp_file)
print('Generation stdout log:', generation_stdout_log)
print('Successful lines:', successful_lines)
print('Failed examples:', failed_examples)
print('Generation latency seconds:', round(generation_latency_seconds, 2))


$ /usr/bin/python3 run_generation.py --in_file /kaggle/working/LongMemEval-Experiment/retrieval_logs/flat-bm25/session/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25 --out_dir /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con --out_file_suffix _20260528-035146_kaggle --model_name cx/gpt-5.2 --model_alias router-gpt-5.2 --retriever_type flat-session --merge_key_expansion_into_value none --topk_context 5 --history_format json --gen_length 300 --useronly false --cot true --con false --openai_base_url https://splashed-nastily-stopped.ngrok-free.dev/v1
Writing command output to: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/run_generation_20260528-035146_kaggle.stdout.log
Expected examples: 500


Generation:   0%|          | 0/500 [00:00<?, ?example/s]

Progress: 0/500 (0.0%) | success=0 | failed=0 | elapsed=0.0 min | ETA=unknown
Stdout log: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/run_generation_20260528-035146_kaggle.stdout.log
Progress: 3/500 (0.6%) | success=3 | failed=0 | elapsed=0.5 min | ETA=82.9 min
Current hypothesis file: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_jsonformat_useronlyfalse_20260528-0351_20260528-035146_kaggle
Stdout log: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/run_generation_20260528-035146_kaggle.stdout.log
Progress: 10/500 (2.0%) | success=10 | failed=0 | elapsed=1.0 min | ETA=49.0 min
Current hypothesis file: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_json

## Step 3: official QA evaluation

The official evaluator asks a metric LLM whether each generated answer is correct. This notebook defaults to `cx/gpt-5.2` through 9router for both generation and judging. For paper-comparable judging, use `METRIC_MODEL_SHORT='gpt-4o'` with an OpenAI endpoint/key.


In [15]:
cmd = [sys.executable, 'evaluate_qa.py', METRIC_MODEL_SHORT, str(hyp_file), str(work_file)]
eval_file = Path(str(hyp_file) + f'.eval-results-{METRIC_MODEL_SHORT}')
evaluation_stdout_log = Path(str(hyp_file) + f'.evaluate_qa-{METRIC_MODEL_SHORT}.stdout.log')
expected_evals = count_lines(hyp_file)

if eval_file.exists():
    eval_file.unlink()

print('$', ' '.join(str(x) for x in cmd))
print('Writing command output to:', evaluation_stdout_log)
print('Expected evaluations:', expected_evals)

evaluation_start_time = time.time()
with evaluation_stdout_log.open('w', encoding='utf-8') as stream:
    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO_DIR / 'src' / 'evaluation'),
        env=env,
        stdout=stream,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pbar = tqdm(total=expected_evals, desc='LLM judge', unit='example')
    last_done = 0
    last_report_ts = 0

    while True:
        done = min(count_lines(eval_file), expected_evals)
        if done > last_done:
            pbar.update(done - last_done)
            last_done = done

        now = time.time()
        if now - last_report_ts >= 30:
            percent = (done / expected_evals * 100) if expected_evals else 0
            elapsed = now - evaluation_start_time
            rate = done / elapsed if elapsed > 0 else 0
            remaining = ((expected_evals - done) / rate) if rate > 0 else None
            eta = f'{remaining/60:.1f} min' if remaining is not None else 'unknown'
            print(f'Judge progress: {done}/{expected_evals} ({percent:.1f}%) | elapsed={elapsed/60:.1f} min | ETA={eta}')
            print('Evaluation log:', eval_file)
            print('Stdout log:', evaluation_stdout_log)
            last_report_ts = now

        if proc.poll() is not None:
            break
        time.sleep(5)

    done = min(count_lines(eval_file), expected_evals)
    if done > last_done:
        pbar.update(done - last_done)
    pbar.close()

evaluation_latency_seconds = time.time() - evaluation_start_time
if proc.returncode != 0:
    print('Evaluation command failed. Last log lines:')
    lines = evaluation_stdout_log.read_text(encoding='utf-8', errors='replace').splitlines()
    for line in lines[-80:]:
        print(line)
    raise RuntimeError(f'evaluate_qa.py failed with exit code {proc.returncode}')

completed_evals = count_lines(eval_file)
if completed_evals != expected_evals:
    raise RuntimeError(f'Evaluation incomplete: expected {expected_evals}, got {completed_evals}. Inspect {evaluation_stdout_log}.')

print('Evaluation log:', eval_file)
print('Evaluation stdout log:', evaluation_stdout_log)
print('Exists:', eval_file.exists())
print('Evaluation latency seconds:', round(evaluation_latency_seconds, 2))


$ /usr/bin/python3 evaluate_qa.py router-gpt-5.2 /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_jsonformat_useronlyfalse_20260528-0351_20260528-035146_kaggle /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned.json
Writing command output to: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_jsonformat_useronlyfalse_20260528-0351_20260528-035146_kaggle.evaluate_qa-router-gpt-5.2.stdout.log
Expected evaluations: 500


LLM judge:   0%|          | 0/500 [00:00<?, ?example/s]

Judge progress: 0/500 (0.0%) | elapsed=0.0 min | ETA=unknown
Evaluation log: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_jsonformat_useronlyfalse_20260528-0351_20260528-035146_kaggle.eval-results-router-gpt-5.2
Stdout log: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_jsonformat_useronlyfalse_20260528-0351_20260528-035146_kaggle.evaluate_qa-router-gpt-5.2.stdout.log
Judge progress: 11/500 (2.2%) | elapsed=0.5 min | ETA=22.2 min
Evaluation log: /kaggle/working/LongMemEval-Experiment/generation_logs/flat-bm25-session/router-gpt-5.2/con/longmemeval_s_cleaned.json_retrievallog_session_flat-bm25_testlog_top5context_jsonformat_useronlyfalse_20260528-0351_20260528-035146_kaggle.eval-results-router-gpt-5.2
Stdout log: /kaggle/working/LongMemEval-Ex

## Aggregate QA and retrieval metrics

The QA summary supports any evaluator alias. Retrieval metrics below match the reporting rule in `run_retrieval.py`: skip abstention items and items without user-side target labels.


In [16]:
eval_rows = [json.loads(line) for line in eval_file.read_text(encoding='utf-8').splitlines() if line.strip()]
ref_rows = {row['question_id']: row for row in json.loads(work_file.read_text(encoding='utf-8'))}
retrieval_rows = [json.loads(line) for line in retrieval_log.read_text(encoding='utf-8').splitlines() if line.strip()]
retrieval_by_id = {row['question_id']: row for row in retrieval_rows}


def get_eval_type(question_id, ref_row):
    if question_id.endswith('_abs') or '_abs' in question_id:
        return 'abstention'
    return ref_row.get('question_type', 'unknown')


def has_user_side_target(row):
    return any(
        ('has_answer' in turn) and bool(turn['has_answer'])
        for session in row.get('haystack_sessions', [])
        for turn in session
        if turn.get('role') == 'user'
    )


def dcg(relevances):
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances))


def compute_session_metrics(retrieval_row, topks=REPORT_TOPKS):
    ranked_ids = [item.get('corpus_id') for item in retrieval_row.get('retrieval_results', {}).get('ranked_items', [])]
    gold_ids = set(retrieval_row.get('answer_session_ids') or [])
    metrics = {'mrr_session': None}
    for k in topks:
        metrics[f'session_recall@{k}'] = None
        metrics[f'session_recall_all@{k}'] = None
        metrics[f'session_ndcg@{k}'] = None
    if not gold_ids:
        return metrics

    first_hit_rank = None
    for idx, corpus_id in enumerate(ranked_ids, start=1):
        if corpus_id in gold_ids:
            first_hit_rank = idx
            break
    metrics['mrr_session'] = 0.0 if first_hit_rank is None else 1.0 / first_hit_rank

    for k in topks:
        top_ids = ranked_ids[:k]
        hit_count = sum(1 for corpus_id in top_ids if corpus_id in gold_ids)
        metrics[f'session_recall@{k}'] = 1.0 if hit_count else 0.0
        metrics[f'session_recall_all@{k}'] = hit_count / len(gold_ids)
        gains = [1 if corpus_id in gold_ids else 0 for corpus_id in top_ids]
        ideal = [1] * min(len(gold_ids), k)
        ideal_dcg = dcg(ideal)
        metrics[f'session_ndcg@{k}'] = 0.0 if ideal_dcg == 0 else dcg(gains) / ideal_dcg
    return metrics


def parse_last_int_after(log_path, prefix):
    if not log_path or not Path(log_path).exists():
        return None
    pattern = re.compile(re.escape(prefix) + r'\s*(\d+)')
    value = None
    for line in Path(log_path).read_text(encoding='utf-8', errors='replace').splitlines():
        match = pattern.search(line)
        if match:
            value = int(match.group(1))
    return value


generation_usage = {
    'total_prompt_tokens': parse_last_int_after(globals().get('generation_stdout_log'), 'Total prompt tokens:'),
    'total_completion_tokens': parse_last_int_after(globals().get('generation_stdout_log'), 'Total completion tokens:'),
}

eval_type_to_scores = {}
question_type_to_scores = {}
abstention_scores = []
question_records = []
for row in eval_rows:
    qid = row['question_id']
    ref = ref_rows[qid]
    score = 1 if row.get('autoeval_label', {}).get('label') else 0
    qtype = ref['question_type']
    eval_type = get_eval_type(qid, ref)
    eval_type_to_scores.setdefault(eval_type, []).append(score)
    question_type_to_scores.setdefault(qtype, []).append(score)
    if eval_type == 'abstention':
        abstention_scores.append(score)

    retrieval_row = retrieval_by_id.get(qid, {})
    rmetrics = compute_session_metrics(retrieval_row)
    score_retrieval = eval_type != 'abstention' and has_user_side_target(retrieval_row)
    topk_recall = rmetrics.get(f'session_recall@{TOPK_CONTEXT}')
    retrieval_miss = None if not score_retrieval or topk_recall is None else topk_recall == 0.0

    record = {
        'question_id': qid,
        'question_type': qtype,
        'eval_type': eval_type,
        'abstention': eval_type == 'abstention',
        'correct': bool(score),
        'retrieval_miss': retrieval_miss,
        'question': ref['question'],
        'answer': ref['answer'],
        'hypothesis': row.get('hypothesis', ''),
    }
    record.update(rmetrics)
    question_records.append(record)

all_scores = [s for scores in eval_type_to_scores.values() for s in scores]
task_scores = [sum(scores) / len(scores) for scores in eval_type_to_scores.values() if scores]

retrieval_metric_records = []
metric_names = sorted({
    metric
    for row in question_records
    for metric in row.keys()
    if metric == 'mrr_session' or metric.startswith('session_')
})
for metric in metric_names:
    values = []
    for row in question_records:
        if row['abstention']:
            continue
        value = row.get(metric)
        if value is not None and not (isinstance(value, float) and math.isnan(value)):
            values.append(value)
    if values:
        retrieval_metric_records.append({
            'granularity': 'session',
            'metric': metric,
            'mean': sum(values) / len(values),
            'n': len(values),
        })

print('Evaluation model:', eval_rows[0]['autoeval_label']['model'] if eval_rows else None)
print('Overall accuracy:', round(sum(all_scores) / len(all_scores), 4) if all_scores else None)
print('Task-averaged accuracy:', round(sum(task_scores) / len(task_scores), 4) if task_scores else None)
print('Abstention accuracy:', round(sum(abstention_scores) / len(abstention_scores), 4) if abstention_scores else None, f'({len(abstention_scores)})')
print('Generation usage:', generation_usage)
print()
print('By eval type:')
for eval_type, scores in sorted(eval_type_to_scores.items()):
    print(f'  {eval_type}: {sum(scores) / len(scores):.4f} ({len(scores)})')


Evaluation model: cx/gpt-5.2
Overall accuracy: 0.736
Task-averaged accuracy: 0.7622
Abstention accuracy: 0.7667 (30)
Generation usage: {'total_prompt_tokens': 8394972, 'total_completion_tokens': 106942}

By eval type:
  abstention: 0.7667 (30)
  knowledge-update: 0.9028 (72)
  multi-session: 0.5455 (121)
  single-session-assistant: 0.8393 (56)
  single-session-preference: 0.6667 (30)
  single-session-user: 0.9219 (64)
  temporal-reasoning: 0.6929 (127)


In [17]:
def fmt_value(value):
    if isinstance(value, float):
        return f'{value:.4f}'
    return 'None' if value is None else str(value)


def print_table(title, rows, columns):
    print(f'\n{title}')
    if not rows:
        print('  No rows.')
        return
    widths = {col: len(col) for col in columns}
    for row in rows:
        for col in columns:
            widths[col] = max(widths[col], len(fmt_value(row.get(col))))
    header = ' | '.join(col.ljust(widths[col]) for col in columns)
    print(header)
    print('-' * len(header))
    for row in rows:
        print(' | '.join(fmt_value(row.get(col)).ljust(widths[col]) for col in columns))


def safe_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return str(dst)


question_count = len(question_records)
correct_count = sum(1 for row in question_records if row['correct'])
abstention_rows = [row for row in question_records if row['abstention']]
failed_rows = [row for row in question_records if not row['correct']]

accuracy_by_eval_type = {
    eval_type: {
        'n': len(scores),
        'accuracy': round(sum(scores) / len(scores), 6) if scores else None,
        'failures': len(scores) - sum(scores),
    }
    for eval_type, scores in sorted(eval_type_to_scores.items())
}
accuracy_by_question_type = {
    qtype: {
        'n': len(scores),
        'accuracy': round(sum(scores) / len(scores), 6) if scores else None,
        'failures': len(scores) - sum(scores),
    }
    for qtype, scores in sorted(question_type_to_scores.items())
}
retrieval_summary = {
    row['metric']: {'mean': round(row['mean'], 6), 'n': row['n']}
    for row in retrieval_metric_records
}

total_prompt_tokens = generation_usage.get('total_prompt_tokens')
total_completion_tokens = generation_usage.get('total_completion_tokens')
generation_seconds = globals().get('generation_latency_seconds')
evaluation_seconds = globals().get('evaluation_latency_seconds')
total_api_seconds = None
if generation_seconds is not None or evaluation_seconds is not None:
    total_api_seconds = (generation_seconds or 0) + (evaluation_seconds or 0)

cost_latency = {
    'experiment_id': EXPERIMENT_ID,
    'examples_evaluated': question_count,
    'total_prompt_tokens': total_prompt_tokens,
    'total_completion_tokens': total_completion_tokens,
    'avg_prompt_tokens': None if total_prompt_tokens is None or not question_count else total_prompt_tokens / question_count,
    'avg_completion_tokens': None if total_completion_tokens is None or not question_count else total_completion_tokens / question_count,
    'generation_latency_seconds': generation_seconds,
    'evaluation_latency_seconds': evaluation_seconds,
    'latency_per_query_seconds': None if total_api_seconds is None or not question_count else total_api_seconds / question_count,
}

a0_hypotheses_file = RESULTS_DIR / f'{EXPERIMENT_ID}_hypotheses.jsonl'
a0_eval_log_file = RESULTS_DIR / f'{EXPERIMENT_ID}_eval_log.jsonl'
a0_retrieval_log_file = RESULTS_DIR / f'{EXPERIMENT_ID}_retrieval_log.jsonl'
a0_summary_file = RESULTS_DIR / f'{EXPERIMENT_ID}_summary.json'
a0_failed_cases_file = RESULTS_DIR / f'{EXPERIMENT_ID}_failed_cases.csv'
a0_cost_latency_file = RESULTS_DIR / f'{EXPERIMENT_ID}_cost_latency.csv'

safe_copy(hyp_file, a0_hypotheses_file)
safe_copy(eval_file, a0_eval_log_file)
safe_copy(retrieval_log, a0_retrieval_log_file)

summary = {
    'experiment_id': EXPERIMENT_ID,
    'dataset': DATASET_NAME,
    'reference_file': str(work_file),
    'retriever': RETRIEVER,
    'granularity': GRANULARITY,
    'topk_context': TOPK_CONTEXT,
    'report_topks': REPORT_TOPKS,
    'reader_model': GEN_MODEL_NAME,
    'reader_model_alias': GEN_MODEL_ALIAS,
    'judge_model': eval_rows[0]['autoeval_label']['model'] if eval_rows else METRIC_MODEL_NAME,
    'judge_model_alias': METRIC_MODEL_SHORT,
    'examples_evaluated': question_count,
    'overall_accuracy': round(correct_count / question_count, 6) if question_count else None,
    'task_averaged_accuracy': round(sum(sum(v) / len(v) for v in eval_type_to_scores.values() if v) / len([v for v in eval_type_to_scores.values() if v]), 6) if eval_type_to_scores else None,
    'abstention_accuracy': None if not abstention_rows else round(sum(1 for row in abstention_rows if row['correct']) / len(abstention_rows), 6),
    'num_failures': len(failed_rows),
    'accuracy_by_eval_type': accuracy_by_eval_type,
    'accuracy_by_question_type': accuracy_by_question_type,
    'retrieval_metrics': retrieval_summary,
    'cost_latency': cost_latency,
    'paths': {
        'retrieval_log': str(a0_retrieval_log_file),
        'hypotheses': str(a0_hypotheses_file),
        'eval_log': str(a0_eval_log_file),
        'summary': str(a0_summary_file),
        'failed_cases': str(a0_failed_cases_file),
        'cost_latency': str(a0_cost_latency_file),
        'generation_stdout_log': str(generation_stdout_log) if 'generation_stdout_log' in globals() else None,
        'evaluation_stdout_log': str(evaluation_stdout_log) if 'evaluation_stdout_log' in globals() else None,
    },
}
a0_summary_file.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')

failed_fieldnames = [
    'question_id', 'eval_type', 'question_type', 'retrieval_miss',
    f'session_recall@{TOPK_CONTEXT}', 'mrr_session', 'question', 'answer', 'hypothesis'
]
with a0_failed_cases_file.open('w', newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=failed_fieldnames)
    writer.writeheader()
    for row in failed_rows:
        writer.writerow({field: row.get(field) for field in failed_fieldnames})

with a0_cost_latency_file.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(cost_latency.keys()))
    writer.writeheader()
    writer.writerow(cost_latency)

summary_rows = [
    {'metric': 'reference_file', 'value': str(work_file)},
    {'metric': 'retrieval_log', 'value': str(a0_retrieval_log_file)},
    {'metric': 'hypothesis_file', 'value': str(a0_hypotheses_file)},
    {'metric': 'evaluation_log', 'value': str(a0_eval_log_file)},
    {'metric': 'summary_file', 'value': str(a0_summary_file)},
    {'metric': 'failed_cases_file', 'value': str(a0_failed_cases_file)},
    {'metric': 'cost_latency_file', 'value': str(a0_cost_latency_file)},
    {'metric': 'evaluation_model', 'value': summary['judge_model']},
    {'metric': 'examples_evaluated', 'value': question_count},
    {'metric': 'overall_accuracy', 'value': summary['overall_accuracy']},
    {'metric': 'task_averaged_accuracy', 'value': summary['task_averaged_accuracy']},
    {'metric': 'abstention_accuracy', 'value': summary['abstention_accuracy']},
    {'metric': 'num_failures', 'value': summary['num_failures']},
    {'metric': 'avg_prompt_tokens', 'value': cost_latency['avg_prompt_tokens']},
    {'metric': 'avg_completion_tokens', 'value': cost_latency['avg_completion_tokens']},
    {'metric': 'latency_per_query_seconds', 'value': cost_latency['latency_per_query_seconds']},
]

by_type_rows = []
for eval_type, item in accuracy_by_eval_type.items():
    by_type_rows.append({
        'eval_type': eval_type,
        'n': item['n'],
        'accuracy': round(item['accuracy'], 4) if item['accuracy'] is not None else None,
        'failures': item['failures'],
    })
by_type_rows.sort(key=lambda row: (row['accuracy'] if row['accuracy'] is not None else -1, -row['n']))

retrieval_rows_print = [
    {
        'granularity': row['granularity'],
        'metric': row['metric'],
        'mean': round(row['mean'], 4),
        'n': row['n'],
    }
    for row in retrieval_metric_records
]
retrieval_rows_print.sort(key=lambda row: (row['granularity'], row['metric']))

print_table('Summary', summary_rows, ['metric', 'value'])
print_table('Accuracy by Eval Type', by_type_rows, ['eval_type', 'n', 'accuracy', 'failures'])
print_table('Retrieval Metrics', retrieval_rows_print, ['granularity', 'metric', 'mean', 'n'])
print('\nA0 artifact files written under:', RESULTS_DIR)



Summary
metric                    | value                                                                                           
----------------------------------------------------------------------------------------------------------------------------
reference_file            | /kaggle/working/LongMemEval-Experiment/data/longmemeval_s_cleaned.json                          
retrieval_log             | /kaggle/working/LongMemEval-Experiment/results/A0_bm25_session_lme_s_cleaned_retrieval_log.jsonl
hypothesis_file           | /kaggle/working/LongMemEval-Experiment/results/A0_bm25_session_lme_s_cleaned_hypotheses.jsonl   
evaluation_log            | /kaggle/working/LongMemEval-Experiment/results/A0_bm25_session_lme_s_cleaned_eval_log.jsonl     
summary_file              | /kaggle/working/LongMemEval-Experiment/results/A0_bm25_session_lme_s_cleaned_summary.json       
failed_cases_file         | /kaggle/working/LongMemEval-Experiment/results/A0_bm25_session_lme_s_cleaned_failed_case

In [18]:
overview_rows = sorted(question_records, key=lambda row: (row['correct'], row['eval_type'], row['question_id']))
preview_rows = overview_rows[:MAX_NOTEBOOK_PREVIEW]
for idx, row in enumerate(preview_rows, start=1):
    preview = row.get('hypothesis', '').replace(chr(10), ' ')[:220]
    print(f"\n[{idx}] correct={'yes' if row['correct'] else 'no'} | eval_type={row['eval_type']} | abstention={row['abstention']}")
    print(f"question_id: {row['question_id']}")
    print(f"session_recall@{TOPK_CONTEXT}: {row.get(f'session_recall@{TOPK_CONTEXT}')} | mrr_session: {row.get('mrr_session')}")
    print(f"question: {row['question']}")
    print(f"answer: {row['answer']}")
    print(f"hypothesis_preview: {preview}")

if len(overview_rows) > len(preview_rows):
    print(f'\nShowing {len(preview_rows)} of {len(overview_rows)} rows. Full records are in {a0_eval_log_file} and {a0_failed_cases_file}.')



[1] correct=no | eval_type=abstention | abstention=True
question_id: 031748ae_abs
session_recall@5: 0.0 | mrr_session: 0.0
question: How many engineers do I lead when I just started my new role as Software Engineer Manager?
answer: The information provided is not enough. You mentioned starting the role as Senior Software Engineer but not Software Engineer Manager.
hypothesis_preview: Step 1: Extract relevant information from the chat history  - Session 3 (2023/05/25):   - You said: “I lead a team of 4 engineers in my new role as Senior Software Engineer.”   - You clarified team size for the outing: “

[2] correct=no | eval_type=abstention | abstention=True
question_id: 09ba9854_abs
session_recall@5: 1.0 | mrr_session: 0.5
question: How much will I save by taking the bus from the airport to my hotel instead of a taxi?
answer: The information provided is not enough. You did not mention how much will the bus take.
hypothesis_preview: Relevant information from the chat history 1) Your tri

In [19]:
failed_rows = [row for row in question_records if not row['correct']]
if not failed_rows:
    print('No failed examples in this evaluation run.')
else:
    preview_rows = failed_rows[:MAX_NOTEBOOK_PREVIEW]
    for idx, row in enumerate(preview_rows, start=1):
        preview = row.get('hypothesis', '').replace(chr(10), ' ')[:600]
        print(f"\nFailed example {idx}")
        print(f"question_id: {row['question_id']}")
        print(f"eval_type: {row['eval_type']}")
        print(f"question_type: {row['question_type']}")
        print(f"retrieval_miss: {row['retrieval_miss']}")
        print(f"question: {row['question']}")
        print(f"answer: {row['answer']}")
        print(f"hypothesis_preview: {preview}")
    if len(failed_rows) > len(preview_rows):
        print(f'\nShowing {len(preview_rows)} of {len(failed_rows)} failed rows. Full failed-case CSV: {a0_failed_cases_file}')



Failed example 1
question_id: 58ef2f1c
eval_type: single-session-user
question_type: single-session-user
retrieval_miss: False
question: When did I volunteer at the local animal shelter's fundraising dinner?
answer: February 14th
hypothesis_preview: Relevant information extracted from the chat history: 1. In Session 5, you said you “really enjoy[ed] the ‘Love is in the Air’ fundraising dinner” and that you “volunteered at [it] back in February.” 2. Later in Session 5, you clarified that this was “back on Valentine’s Day.”  Reasoning to answer the question: - The only fundraising dinner you explicitly mentioned volunteering at is the “Love is in the Air” fundraising dinner. - You identified the volunteering time as Valentine’s Day (which is in February).  Answer: You volunteered at the local animal shelter’s fundraising dinner on Valentine

Failed example 2
question_id: 75499fd8
eval_type: single-session-user
question_type: single-session-user
retrieval_miss: True
question: What breed 

## Optional: long-context baseline

This baseline provides the full recent history to the reader. It is expensive on `longmemeval_s_cleaned.json` and not practical for `longmemeval_m_cleaned.json` with normal Kaggle limits. Enable it only after the smoke test succeeds.


In [20]:
RUN_LONG_CONTEXT_BASELINE = False

if RUN_LONG_CONTEXT_BASELINE:
    full_out_dir = REPO_DIR / 'generation_logs' / 'full-history-session' / GEN_MODEL_ALIAS / 'con'
    full_out_dir.mkdir(parents=True, exist_ok=True)
    full_suffix = f'_{run_id}_kaggle_fullhistory'
    cmd = [
        sys.executable, 'run_generation.py',
        '--in_file', str(work_file),
        '--out_dir', str(full_out_dir),
        '--out_file_suffix', full_suffix,
        '--model_name', GEN_MODEL_NAME,
        '--model_alias', GEN_MODEL_ALIAS,
        '--retriever_type', 'orig-session',
        '--merge_key_expansion_into_value', 'none',
        '--topk_context', '1000',
        '--history_format', HISTORY_FORMAT,
        '--useronly', USERONLY,
        '--cot', 'true',
        '--con', 'false',
    ]
    if OPENAI_BASE_URL:
        cmd.extend(['--openai_base_url', OPENAI_BASE_URL])

    expected_full_examples = len(work_data)
    full_stdout_log = full_out_dir / f'run_generation{full_suffix}.stdout.log'
    print('$', ' '.join(str(x) for x in cmd))
    print('Writing command output to:', full_stdout_log)
    print('Expected examples:', expected_full_examples)

    full_start_time = time.time()
    with full_stdout_log.open('w', encoding='utf-8') as stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(REPO_DIR / 'src' / 'generation'),
            env=env,
            stdout=stream,
            stderr=subprocess.STDOUT,
            text=True,
        )

        pbar = tqdm(total=expected_full_examples, desc='Full-history generation', unit='example')
        last_done = 0
        last_report_ts = 0
        full_hyp_file = None

        while True:
            hyp_files = sorted(full_out_dir.glob(f'*{full_suffix}'), key=lambda path: path.stat().st_mtime)
            if hyp_files:
                full_hyp_file = hyp_files[-1]
            successful = count_lines(full_hyp_file) if full_hyp_file else 0
            failed = count_occurrences(full_stdout_log, 'One exception captured')
            done = min(successful + failed, expected_full_examples)
            if done > last_done:
                pbar.update(done - last_done)
                last_done = done

            now = time.time()
            if now - last_report_ts >= 30:
                percent = (done / expected_full_examples * 100) if expected_full_examples else 0
                elapsed = now - full_start_time
                rate = done / elapsed if elapsed > 0 else 0
                remaining = ((expected_full_examples - done) / rate) if rate > 0 else None
                eta = f'{remaining/60:.1f} min' if remaining is not None else 'unknown'
                print(f'Full-history progress: {done}/{expected_full_examples} ({percent:.1f}%) | elapsed={elapsed/60:.1f} min | ETA={eta}')
                print('Stdout log:', full_stdout_log)
                last_report_ts = now

            if proc.poll() is not None:
                break
            time.sleep(5)

        successful = count_lines(full_hyp_file) if full_hyp_file else 0
        failed = count_occurrences(full_stdout_log, 'One exception captured')
        done = min(successful + failed, expected_full_examples)
        if done > last_done:
            pbar.update(done - last_done)
        pbar.close()

    if proc.returncode != 0:
        print('Full-history generation failed. Last log lines:')
        lines = full_stdout_log.read_text(encoding='utf-8', errors='replace').splitlines()
        for line in lines[-80:]:
            print(line)
        raise RuntimeError(f'full-history run_generation.py failed with exit code {proc.returncode}')
    if not full_hyp_file or count_lines(full_hyp_file) != expected_full_examples:
        raise RuntimeError(f'Full-history generation incomplete. Inspect {full_stdout_log}.')
    print('Full-history hypothesis file:', full_hyp_file)
else:
    print('Skipped. Set RUN_LONG_CONTEXT_BASELINE=True to run this optional baseline.')


Skipped. Set RUN_LONG_CONTEXT_BASELINE=True to run this optional baseline.


## Optional: package outputs

Run this cell to create a downloadable archive under Kaggle output.


In [21]:
archive = ROOT / 'longmemeval_A0_benchmark_outputs.tar.gz'
archive_items = [name for name in ['retrieval_logs', 'generation_logs', 'results'] if (REPO_DIR / name).exists()]
if archive_items:
    run_cmd(['tar', '-czf', str(archive), *archive_items], cwd=REPO_DIR, env=env, check=False)
    print('Archive:', archive)
    print('Archived folders:', archive_items)
else:
    print('No output folders found to archive yet.')


$ tar -czf /kaggle/working/longmemeval_A0_benchmark_outputs.tar.gz retrieval_logs generation_logs results
Archive: /kaggle/working/longmemeval_A0_benchmark_outputs.tar.gz
Archived folders: ['retrieval_logs', 'generation_logs', 'results']
